# BISINDO A-Z: Rhio + Sanjaya, training GPU Colab

Upload SATU file notebook ini ke Google Colab. Pilih **Runtime > Change runtime type > T4 GPU** (atau GPU lain yang tersedia), kemudian jalankan sel berurutan. Notebook memeriksa CUDA sebelum mengunduh dataset. GPU tidak dijamin tersedia oleh Colab.

Classifier memakai **PyTorch MLP pada CUDA**. Unduhan foto dan ekstraksi MediaPipe Tasks Vision browser masih CPU; GPU tidak otomatis mempercepat dua tahap tersebut. Dataset kecil juga belum tentu lebih cepat dengan GPU. Tidak ada webcam yang direkam.

Kedua dataset diunduh otomatis: Rhio (MIT) dan Samuel Ady Sanjaya, 18 Oktober 2024, v1, DOI 10.17632/ywnjpbcz8m.1 (CC BY 4.0). Hanya Original Images Sanjaya; versi resized/binary tidak digabung. Label mengikuti XML/folder penerbit, bukan validasi semua varian gesture. Tinjau contoh kedua sumber.

Training tetap memakai extractor TypeScript 52 fitur dan MediaPipe Tasks Vision 1.0.1 yang sama dengan website. Standardization model dipasang setelah extractor, dipelajari hanya dari train split dan disertakan pada ekspor. Tidak ada flip/augmentation. Split per foto asli, bukan per orang karena identitas signer/session tidak tersedia.

Output ZIP berisi model.pt, model.json MLP, evaluasi, confusion matrix, sumber dan atribusi. **Format MLP berbeda dari Random Forest website saat ini.** Model ini kandidat; integrasi runtime browser, konten A-Z dan uji kamera masih diperlukan. Foto statis tidak cukup untuk memvalidasi huruf yang membutuhkan gerakan.

Sumber: https://github.com/rhiosutoyo/Indonesian-Sign-Language-BISINDO-Hand-Sign-Detection-Dataset dan https://data.mendeley.com/datasets/ywnjpbcz8m/1 . GPU: https://research.google.com/colaboratory/faq.html .


In [ ]:
import os, json, pathlib, subprocess, urllib.request, hashlib, tarfile, shutil, base64
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Pilih Runtime > Change runtime type > T4 GPU sebelum melanjutkan.')
print('GPU:', torch.cuda.get_device_name(0), '| PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
ROOT = pathlib.Path('/content/bisindo-alphabet-training')
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
def run(*args):
    print('Menjalankan:', ' '.join(args), flush=True)
    tail = []
    with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
            tail.append(line)
            tail = tail[-30:]
        status = process.wait()
    if status:
        raise RuntimeError('Perintah gagal: ' + ' '.join(args) + '\n' + ''.join(tail))
version = '24.21.0'
archive = f'node-v{version}-linux-x64.tar.xz'
url = f'https://nodejs.org/dist/v{version}/'
if not pathlib.Path('node/bin/node').exists():
    urllib.request.urlretrieve(url + archive, archive)
    checksums = urllib.request.urlopen(url + 'SHASUMS256.txt').read().decode()
    expected = next(line.split()[0] for line in checksums.splitlines() if line.split()[-1] == archive)
    assert hashlib.sha256(pathlib.Path(archive).read_bytes()).hexdigest() == expected, 'Node checksum mismatch'
    with tarfile.open(archive) as tar:
        tar.extractall('.', filter='data')
    pathlib.Path(f'node-v{version}-linux-x64').rename('node')
os.environ['PATH'] = str(ROOT / 'node/bin') + ':' + os.environ['PATH']
run('node', '--version')


In [ ]:
# Source snapshot embedded: no private repository credentials needed.
bundle = json.loads(base64.b64decode('...'))
for name, content in bundle.items():
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
package = {'private': True, 'type': 'module', 'dependencies': {'@mediapipe/tasks-vision':'1.0.1', '@playwright/test':'1.63.0', 'typescript':'5.9.3'}}
pathlib.Path('package.json').write_text(json.dumps(package))
run('npm', 'install', '--no-audit', '--no-fund')
run('npx', 'playwright', 'install', '--with-deps', 'chromium')
provenance = json.loads(pathlib.Path('public/models/mediapipe/provenance.json').read_text())
for asset in provenance['assets']:
    target = pathlib.Path(asset['path']); target.parent.mkdir(parents=True, exist_ok=True)
    if target.suffix == '.task':
        if not target.exists(): urllib.request.urlretrieve(provenance['modelSource'], target)
    else:
        shutil.copyfile(pathlib.Path('node_modules/@mediapipe/tasks-vision/wasm') / target.name, target)
    assert hashlib.sha256(target.read_bytes()).hexdigest() == asset['sha256'], str(target) + ' checksum mismatch'
print('Runtime dan model MediaPipe siap; checksum sesuai website.')


## 1. Unduh dan gabungkan dua dataset
Semua Aâ€“Z dari kedua sumber. Sel ini memerlukan jaringan dan ruang disk untuk foto asli. File yang sudah cocok checksum tidak diunduh ulang. Jika layanan penerbit gagal, jalankan kembali sel ini. Tidak ada data kamera pribadi.

In [ ]:
run('node', 'scripts/colab/prepare-combined.mjs')
manifest = json.loads(pathlib.Path('ml/rhio/manifest.json').read_text())
from collections import Counter
print(Counter((r['sourceId'], r['split']) for r in manifest['samples']))


## 2. Tinjau contoh kedua sumber
Periksa apakah label dan bentuk dalam kedua sumber cocok. Foto di bawah hanya contoh, bukan validasi semua data. Konflik varian harus diselesaikan sebelum model dipakai untuk menyatakan gesture benar.

In [ ]:
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
LETTER = 'C'  # Ubah menjadi huruf A-Z untuk memeriksa sumber.
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for row, source in enumerate(['rhio-bisindo-2024', 'sanjaya-bisindo-alphabet-2024-v1']):
    samples = [r for r in manifest['samples'] if r['label'] == LETTER and r['sourceId'] == source][:3]
    for ax, sample in zip(axes[row], samples):
        with Image.open(ROOT / '.tools/bisindo-dataset' / sample['id']) as im:
            ax.imshow(ImageOps.exif_transpose(im))
        ax.set_title(source.split('-')[0] + ' / ' + LETTER); ax.axis('off')
plt.tight_layout(); plt.show()


## 3. Ekstrak titik dengan pipeline browser
Mode VIDEO memakai tiga timestamp per foto untuk warm-up. Evaluasi hanya memakai tick terakhir per foto. Tangan ambigu/tidak terlihat dan fitur tidak valid ditolak. Tidak ada webcam yang digunakan.

In [ ]:
run('node', 'scripts/extract-rhio-landmarks.mjs')
features = json.loads(pathlib.Path('.tools/bisindo-dataset/features.json').read_text())
print(Counter(r['rejection'] or 'USABLE' for r in features['rows'] if r['tick'] == 2))


## 4. Training classifier dengan GPU
MLP 52 -> 128 -> 64 -> 26 memakai CUDA, mini-batch dan early stopping. Model terbaik dipilih dari validation loss; threshold hanya dikalibrasi pada validation, lalu test dievaluasi sekali. Data tidak cukup setelah ekstraksi menyebabkan error coverage yang jelas. Hasil dapat UNCERTAIN; probability bukan kebenaran gesture. Waktu training dan nama GPU dicatat.


In [ ]:
# GPU classifier training; landmark extraction remains the exact browser pipeline.
import copy, random, time
import numpy as np
import torch
from torch import nn
import pandas as pd

REQUIRE_GPU = True  # Explicitly set False only for a CPU experiment.
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError('GPU belum aktif. Pilih Runtime > Change runtime type > T4 GPU, lalu jalankan ulang notebook.')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Training device:', device, torch.cuda.get_device_name(0) if device.type == 'cuda' else '')
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
torch.backends.cuda.matmul.allow_tf32 = False
random.seed(42); np.torch.backends.cuda.matmul.allow_tf32 = False
random.seed(42); torch.manual_seed(42)
if device.type == 'cuda': torch.cuda.manual_seed_all(42)

# Recompute every usable vector against the embedded TypeScript feature contract.
run('node', '--input-type=module', '-e', "import {readFile} from 'node:fs/promises'; import {loadTrainingContract} from './scripts/load-training-contract.mjs'; const r=async p=>JSON.parse(await readFile(p,'utf8')); const c=await loadTrainingContract(); c.validateTrainingData(await r('ml/rhio/manifest.json'),await r('.tools/bisindo-dataset/features.json'),c.featureSchema,c.extractFeatures); console.log('Feature parity passed');")
dataset = json.loads(pathlib.Path('.tools/bisindo-dataset/features.json').read_text())
manifest = json.loads(pathlib.Path('ml/rhio/manifest.json').read_text())
labels = sorted(set(r['label'] for r in manifest['samples']))
rows = [r for r in dataset['rows'] if r['tick'] == 2 and r['vector'] is not None]
parts = {s: [r for r in rows if r['split'] == s] for s in ['train','validation','test']}
coverage = [{'label': l, **{s: sum(r['label']==l for r in parts[s]) for s in parts}} for l in labels]
pathlib.Path('output').mkdir(exist_ok=True)
pathlib.Path('output/coverage.json').write_text(json.dumps(coverage, indent=2))
missing = [r['label'] for r in coverage if r['train'] < 2 or r['validation'] < 1 or r['test'] < 1]
if missing: raise RuntimeError('Kurang data setelah ekstraksi: '+', '.join(missing)+'. Lihat output/coverage.json.')
arrays = {s: np.array([r['vector'] for r in parts[s]], dtype=np.float32) for s in parts}
# Standardization fitted on TRAIN only; parameters are exported with model weights.
mean = arrays['train'].mean(axis=0)
scale = arrays['train'].std(axis=0)
scale[scale < 1e-6] = 1.0
xs = {s: torch.tensor((arrays[s]-mean)/scale, device=device) for s in parts}
ys = {s: torch.tensor([labels.index(r['label']) for r in parts[s]], dtype=torch.long, device=device) for s in parts}
model = nn.Sequential(nn.Linear(52,128), nn.ReLU(), nn.Linear(128,64), nn.ReLU(), nn.Linear(64,len(labels))).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
weights = 1.0 / torch.bincount(ys['train'], minlength=len(labels)).float()
loss_fn = nn.CrossEntropyLoss(weight=weights / weights.mean())
best_loss, best_state, stale, best_epoch = float('inf'), None, 0, 0
MAX_EPOCHS, PATIENCE, BATCH_SIZE = 250, 25, 256
started = time.monotonic()
for epoch in range(MAX_EPOCHS):
    model.train()
    permutation = torch.randperm(len(xs['train']), device=device)
    for indices in permutation.split(BATCH_SIZE):
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(model(xs['train'][indices]), ys['train'][indices])
        loss.backward(); optimizer.step()
    model.eval()
    with torch.no_grad(): val_loss = nn.functional.cross_entropy(model(xs['validation']), ys['validation']).item()
    if not np.isfinite(val_loss): raise RuntimeError('Non-finite validation loss')
    if val_loss < best_loss - 1e-5:
        best_loss, best_state, stale, best_epoch = val_loss, copy.deepcopy(model.state_dict()), 0, epoch+1
    else: stale += 1
    if epoch == 0 or (epoch+1)%10 == 0: print(f'Epoch {epoch+1}: validation loss={val_loss:.4f}', flush=True)
    if stale >= PATIENCE: break
model.load_state_dict(best_state); model.eval()
with torch.no_grad(): probabilities = {s: torch.softmax(model(xs[s]), dim=1).cpu().numpy() for s in ['validation','test']}

# Distance rejection uses original schema, same presence mask, train/validation only.
envelopes = []
for label in labels:
    positive = [r for r in parts['train'] if r['label']==label]
    distances = []
    for row in positive + [r for r in parts['validation'] if r['label']==label]:
        candidates = [r['vector'] for r in positive if r['groupId'] != row['groupId'] and r['vector'][50:52] == row['vector'][50:52]]
        if candidates: distances.append(float(np.sqrt(np.mean((np.array(candidates)-row['vector'])**2, axis=1)).min()))
    if not distances: raise RuntimeError('No distance coverage: '+label)
    envelopes.append({'label':label,'maxDistance':float(np.quantile(distances,.95)), 'vectors':[r['vector'] for r in positive]})
def predict(split, threshold):
    accepted = []
    for row, scores in zip(parts[split], probabilities[split]):
        order = np.argsort(scores); top = int(order[-1]); envelope = envelopes[top]
        candidates = [v for v in envelope['vectors'] if v[50:52] == row['vector'][50:52]]
        nearest = float(np.sqrt(np.mean((np.array(candidates)-row['vector'])**2,axis=1)).min()) if candidates else float('inf')
        accepted.append(top if scores[top]>=threshold and scores[top]-scores[order[-2]]>=.1 and nearest<=envelope['maxDistance'] else len(labels))
    return np.array(accepted)
truth = {s: ys[s].cpu().numpy() for s in ['validation','test']}
calibrations=[]
for threshold in [.5,.55,.6,.65,.7,.75,.8,.85,.9,.95,1.0]:
    pred=predict('validation',threshold)
    correct=int((pred==truth['validation']).sum()); wrong=int(((pred!=truth['validation'])&(pred<len(labels))).sum())
    calibrations.append({'threshold':threshold,'correct':correct,'wrong':wrong,'utility':correct-4*wrong})
calibration=max(calibrations,key=lambda c:(c['utility'],-c['wrong'],c['threshold']))
pred=predict('test',calibration['threshold'])
confusion=np.zeros((len(labels),len(labels)+1),dtype=int)
for actual, guessed in zip(truth['test'],pred): confusion[actual,guessed]+=1

# Portable candidate weights; not the existing ml-random-forest JSON format.
layers=[{'weight': layer.weight.detach().cpu().tolist(), 'bias':layer.bias.detach().cpu().tolist()} for layer in model if isinstance(layer,nn.Linear)]
payload={'id':'combined-alphabet-mlp-candidate-v1','version':'1.0.0','status':'EXPERIMENTAL','deploymentReady':False,'labels':labels,'runtime':{'name':'dense-relu-mlp-json','version':'1','weightLayout':'output-by-input','hiddenActivation':'relu','outputActivation':'softmax'},'featureSchema':dataset['featureSchema'],'landmarker':dataset['trackingConfig'],'sources':manifest['sources'],'inputTransform':{'kind':'standardize','mean':mean.tolist(),'scale':scale.tolist()},'layers':layers,'threshold':calibration['threshold'],'minMargin':.1,'envelopes':envelopes}
serialized=json.dumps(payload,allow_nan=False)
pathlib.Path('output/model.json').write_text(serialized)
exported=json.loads(serialized)
logits=(arrays['test']-np.array(exported['inputTransform']['mean']))/np.array(exported['inputTransform']['scale'])
for i,layer in enumerate(exported['layers']):
    logits=logits@np.array(layer['weight']).T+np.array(layer['bias'])
    if i<len(exported['layers'])-1: logits=np.maximum(logits,0)
restored=np.exp(logits-logits.max(axis=1,keepdims=True));restored/=restored.sum(axis=1,keepdims=True)
assert np.allclose(restored,probabilities['test'],atol=1e-4), 'Export parity failed'
torch.save({ 'state_dict':{k:v.cpu() for k,v in model.state_dict().items()},'metadata':payload },'output/model.pt')
report={'modelSha256':hashlib.sha256(serialized.encode()).hexdigest(),'training':{'device':str(device),'gpu':torch.cuda.get_device_name(0) if device.type=='cuda' else None,'torch':str(torch.__version__),'cuda':torch.version.cuda,'bestEpoch':best_epoch,'seconds':time.monotonic()-started},'coverage':coverage,'calibration':calibration,'testConfusion':{'rows':labels,'columns':labels+['UNCERTAIN'],'values':confusion.tolist()},'perLetter':[{'label':label,'tested':int(confusion[i].sum()),'matched':int(confusion[i,i]),'uncertain':int(confusion[i,-1]),'wrong':int(confusion[i].sum()-confusion[i,i]-confusion[i,-1])} for i,label in enumerate(labels)],'limitations':['Static photo-label experiment; not dynamic sign validation.','Signer/session independence unknown.','No live or unknown-pose acceptance evaluation.','MLP format requires browser runtime integration and testing; do not replace the current C/L/O forest.'],'golden':[{'id':r['id'],'label':r['label'],'frame':r['frame'],'vector':r['vector'],'probabilities':scores.tolist(),'predicted':labels[int(p)] if p<len(labels) else None} for r,scores,p in zip(parts['test'],probabilities['test'],pred)]}
pathlib.Path('output/evaluation.json').write_text(json.dumps(report,indent=2,allow_nan=False))
display(pd.DataFrame(report['perLetter']))
print('Training selesai. Export parity passed.', report['training'])


In [ ]:
import numpy as np
matrix = np.array(report['testConfusion']['values'])
fig, ax = plt.subplots(figsize=(15, 12))
chart = ax.imshow(matrix, cmap='Blues')
ax.set_xticks(range(len(report['testConfusion']['columns'])), report['testConfusion']['columns'], rotation=90)
ax.set_yticks(range(len(report['testConfusion']['rows'])), report['testConfusion']['rows'])
ax.set_xlabel('Prediksi'); ax.set_ylabel('Label sumber'); fig.colorbar(chart)
plt.tight_layout(); plt.savefig('output/confusion.png'); plt.show()
# Source breakdown exposes differences hidden by combined totals.
by_id = {r['id']: r for r in manifest['samples']}
breakdown = Counter()
for row in report['golden']:
    outcome = 'matched' if row['predicted'] == row['label'] else 'uncertain' if row['predicted'] is None else 'wrong'
    breakdown[(by_id[row['id']]['sourceId'], row['label'], outcome)] += 1
source_report = [{'source': s, 'letter': l, 'outcome': o, 'count': n} for (s,l,o),n in sorted(breakdown.items())]
pd.DataFrame(source_report).to_csv('output/evaluation-by-source.csv', index=False)
display(pd.DataFrame(source_report))


## 5. Download hasil
ZIP tidak berisi foto. model.pt adalah checkpoint PyTorch; model.json adalah bobot MLP + normalisasi + rejection envelope. Format ini **bukan** format forest website. Simpan hasil untuk integrasi runtime dan uji browser berikutnya, jangan langsung mengganti model produksi.


In [ ]:
shutil.copyfile('ml/rhio/manifest.json', 'output/manifest.json')
shutil.copyfile('public/models/mediapipe/provenance.json', 'output/mediapipe-provenance.json')
shutil.copyfile('package-lock.json', 'output/package-lock.json')
pathlib.Path('output/source-snapshot.json').write_text(json.dumps(bundle))
shutil.make_archive('/content/bisindo-alphabet-results', 'zip', 'output')
from google.colab import files
files.download('/content/bisindo-alphabet-results.zip')
